# A learning rate for every arm, chosen on the dev split

Report 08's SIB-200 table compares five models. Two of them were swept over five learning rates
and the best cell quoted; the other three were run at one learning rate each.

| model | learning rates run | reported | how the cell was chosen |
|---|---|---|---|
| mmBERT | 5 | 0.574 | best of five, on the test set |
| XLM-R | 5 | 0.408 | best of five, on the test set |
| from-scratch, ours | **1** | 0.632 | `FT_LR`, the default |
| our architecture, untrained | **1** | 0.403 | `FT_LR_SCRATCH` |
| XLM-R's architecture, untrained | **1** | 0.369 | 2e-5 |

Best-of-five is a selection. Applying it to two rows and not the other three tilts every
comparison that crosses the line, and it tilts them in **opposite directions**:

- **ours vs mmBERT (0.632 vs 0.574)** — our unswept default against mmBERT's best cell. Biased
  *against* us, so the win is conservative. That is the safe direction, but the margin is not the
  one a symmetric comparison would give.
- **XLM-R vs its own untrained control (+0.039)** — XLM-R's best-of-five against a control run
  once. Biased *in XLM-R's favour*, so **+0.039 is an upper bound**. Sweep the control and it can
  only rise.

Sweeping the three missing arms fixes that. It does not fix the second problem, which
`exp_budget_matched_baselines.ipynb` already flagged in its own head-to-head cell:

> *That is a choice made on the test set — SIB-200 ships a 99-item validation split, so select
> there instead before quoting a single number in a writeup.*

Nothing has used that split. So this notebook does both: **every arm gets the same five learning
rates, ranked on the 99 dev items, and only the winner is then scored on the 204 test items.**

Needs `ft_api` ≥ (1, 4) for `eval_split`. Dev-scored records are tagged `_onval` and
`ft.results()` excludes them by default — a cell selected on the items it is scored on is not a
reportable number, and mixing the two tables is how one ends up on a poster.

In [ ]:
import os, sys
REPO = '/content/WashingtonCsed504'
FORK = 'https://github.com/patlkwok/WashingtonCsed504.git'   # YOUR fork, not upstream
if not os.path.exists(REPO):
    !git clone -q {FORK} {REPO}
sys.path.insert(0, f'{REPO}/src/a2-nlp')
import session; factory = session.start(prepare=False)

In [ ]:
import importlib, time
import numpy as np
import ft_api as ft
importlib.reload(ft)

assert ft.API_VERSION >= (1, 4), (
    f'ft_api is {ft.API_VERSION}; this notebook needs (1, 4) for eval_split. '
    'Pull the fork and reload -- %autoreload does not work on Python 3.12.')

GPU = ft.gpu_name()
print('ft_api', ft.API_VERSION, '| GPU', GPU)
if 'A100' not in GPU and 'PRO 6000' not in GPU:
    print('  NOTE: the timings below were measured on an A100/Blackwell. This is neither; '
          'expect them to differ by the multipliers in CLAUDE.md 2.1.')

sib = ft.load_sib200('yor_Latn')
print('dev items:', len(sib['validation']['text']), '| test items:', len(sib['test']['text']))

## The asymmetry, measured rather than asserted

Recomputed from `runs/` so it cannot go stale in this markdown the way a pasted table would.

In [ ]:
STEPS = 1056

rows = [r for r in ft.results(task='sib200') if r['steps'] == STEPS]
by_model = {}
for r in rows:
    by_model.setdefault(r['model_slug'], []).append(r)

print(f'{"model":<32}{"LRs run":>8}{"best":>8}{"at lr":>9}   per-seed s')
for slug, rs in sorted(by_model.items(), key=lambda kv: -max(r['mean'] for r in kv[1])):
    best = max(rs, key=lambda r: r['mean'])
    print(f'{slug:<32}{len(rs):>8}{best["mean"]:>8.3f}{best["lr"]:>9.0e}   '
          f'{np.mean([r["seconds_per_seed"] for r in rs]):.0f}')

unswept = [s for s, rs in by_model.items() if len(rs) < 5]
print(f'\nunswept arms: {unswept}')

## The checkpoints

Three of the five are free. `random_init` and `random_init_like` build an untrained model from a
config in seconds, so both controls regenerate on any runtime. The two hub baselines download.

**The from-scratch checkpoint does not.** `runs/<tag>/` directories are not tracked in git
(CLAUDE.md 6.1 — they are hundreds of MB), so `yor_64M_62.5k_s0` exists only on the machine that
trained it. Three ways to get it, in order of cost:

1. **Drive**, if Jeffrey has put it there — set `DRIVE_CKPT` and mount. **Much the best option.**
2. Ask Jeffrey to run this notebook's sweep on the card that already has the checkpoint.
3. **Re-pretrain it here**, at 64M tokens for 62,500 steps — ~40 min on a Blackwell, so
   **1.5–2 h on an A100**, on top of the ~100 min the sweep itself costs.

**Re-pretraining does NOT reproduce `yor_64M_62.5k_s0`, and the tag must not say it does.**
Same corpus, same seed and same steps on a different GPU is a different model — nondeterministic
kernels, and this one is a 33.8M model whose seed spread at this cell is 0.103. Three things go
wrong if it inherits the name:

- `runs/yor_64M_62.5k_s0_result.json` is **committed** (val 2.315, 2,409 s on Jeffrey's card).
  `pretrain` would overwrite a scientific record from PR #32 with a run from other hardware.
- The from-scratch **test** row already exists at 0.632, measured on Jeffrey's checkpoint.
  `ft.evaluate(reuse=True)` would hand it straight back while the dev sweep ran on the new
  checkpoint — two different models under one name, inside one comparison.
- Nothing would announce either.

So the retrain path writes `yor_64M_62.5k_s0_local`. Distinct pretraining record, distinct
`model_slug`, nothing reused, nothing overwritten — and the comparison stays internally consistent
because *every* cell in it, dev and test, is then measured on the same checkpoint.

The cell fails loudly rather than skipping the arm. A sweep silently missing the one model the
poster's headline rests on is worse than no sweep.

In [ ]:
SCRATCH_TAG   = 'yor_64M_62.5k_s0'        # corpus yor, 64M tokens, 62.5k steps, preset poc, seed 0
DRIVE_CKPT    = ''                        # e.g. '/content/drive/MyDrive/csed504/yor_64M_62.5k_s0'
REPRETRAIN    = False                     # True -> retrain it here as _local (1.5-2 h on an A100)
ALLOW_PARTIAL = False                     # True -> sweep the other four arms without it

MMBERT = 'jhu-clsp/mmBERT-base'
XLMR   = 'FacebookAI/xlm-roberta-base'

local = os.path.join(factory.RUNS, SCRATCH_TAG)

if os.path.exists(os.path.join(local, 'config.json')):
    scratch = local
    print(f'from-scratch checkpoint: {scratch} (already on this runtime)')
elif DRIVE_CKPT and os.path.exists(os.path.join(DRIVE_CKPT, 'config.json')):
    scratch = DRIVE_CKPT
    print(f'from-scratch checkpoint: {scratch} (from Drive)')
elif REPRETRAIN:
    # NOT 'yor_64M_62.5k_s0'. That record is committed and its downstream row already exists;
    # see the markdown above for what reusing the name would silently do.
    factory = session.start(corpus='yor')                 # verifies fingerprint 15abd33de5af
    rec = factory.pretrain('yor', tokens=64_000_000, steps=62_500, seed=0, preset='poc',
                           tag='yor_64M_62.5k_s0_local')
    scratch = os.path.join(factory.RUNS, rec['tag'])
    print(f'from-scratch checkpoint: {scratch} (RETRAINED HERE, val {rec["val_loss"]:.3f} '
          f'against 2.315 on Jeffrey\'s card)')
    print('  every from-scratch cell below is measured on THIS checkpoint; the committed 0.632 '
          'row is not\n  comparable with them and is excluded from the table.')
elif ALLOW_PARTIAL:
    scratch = None
    print(f'*** {SCRATCH_TAG} NOT FOUND -- sweeping the other four arms without it. ***\n'
          '    The XLM-R-vs-its-own-control correction is complete without this arm; the\n'
          '    ours-vs-mmBERT comparison is NOT, and the table will say so.')
else:
    raise FileNotFoundError(
        f'{SCRATCH_TAG} is not on this runtime and no DRIVE_CKPT was given.\n'
        'It is the model the SIB-200 headline rests on, so this notebook will not quietly '
        'produce a table\nwithout it. Set DRIVE_CKPT, or ALLOW_PARTIAL = True to sweep the other '
        'four arms, or REPRETRAIN = True,\nor ask Jeffrey to run this on the card that already '
        'has the checkpoint.')

# Whether the from-scratch arm is the SAME checkpoint report 08 measured. If it is not, the
# 0.632 row is a different model and the table below must not difference against it.
SCRATCH_IS_CANONICAL = bool(scratch) and not scratch.endswith('_local')

# Both controls are cheap to rebuild and are byte-identical to the ones already measured.
factory = session.start(corpus='yor')
rand_ours = factory.random_init('yor')
rand_xlmr = factory.random_init_like(XLMR)

ARMS = [('mmBERT',          MMBERT),
        ('XLM-R',           XLMR),
        ('untrained ours',  rand_ours),
        ('untrained XLM-R', rand_xlmr)]
if scratch:
    ARMS.insert(2, ('from-scratch ours', scratch))

print()
for name, path in ARMS:
    print(f'  {name:<20} {path}')
print(f'\n{len(ARMS)} of 5 arms' + ('' if scratch else '  <-- PARTIAL'))

## The dev sweep

Five learning rates, every arm, scored on the 99 dev items. **Three seeds here, not five** — this
pass only has to *rank* the learning rates, and the reported number comes from a separate
five-seed run on test below. Three is the project minimum (CLAUDE.md 6.4).

Costed from the `seconds_per_seed` already on the records: roughly **90 minutes on an A100**.
`reuse=True` means an interrupted session resumes rather than restarting.

In [ ]:
LRS       = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
DEV_SEEDS = (0, 1, 2)
REUSE     = True
QUICK     = False        # True -> 2 LRs, to check the plumbing before committing 90 minutes

lrs = LRS[1:3] if QUICK else LRS
t0 = time.time()

for name, path in ARMS:
    print(f'\n{name}')
    for lr in lrs:
        ft.evaluate(path, task='sib200', lr=lr, steps=STEPS, seeds=DEV_SEEDS,
                    data=sib, reuse=REUSE, label=f'{name} lr{lr:g}',
                    eval_split='validation')

print(f'\ndev sweep: {(time.time() - t0) / 60:.1f} min')

## Select on dev, report on test

The winner of each arm — chosen on the 99 dev items it will *not* be reported on — is then run at
five seeds on test. Where that cell already exists from earlier work it is reused, so this pass is
short.

In [ ]:
SEEDS = (0, 1, 2, 3, 4)

dev = ft.results(task='sib200', eval_split='validation')
picked = {}
for name, path in ARMS:
    rs = [r for r in dev if r['model'] == path and r['steps'] == STEPS]
    if not rs:
        print(f'{name}: no dev cells -- run the sweep above'); continue
    win = max(rs, key=lambda r: r['mean'])
    picked[name] = (path, win['lr'], win['mean'])
    ranked = ', '.join(f'{r["lr"]:g}:{r["mean"]:.3f}' for r in sorted(rs, key=lambda r: -r['mean']))
    print(f'{name:<20} dev pick lr {win["lr"]:g}  ({ranked})')

print()
final = {}
for name, (path, lr, _) in picked.items():
    final[name] = ft.evaluate(path, task='sib200', lr=lr, steps=STEPS, seeds=SEEDS,
                              data=sib, reuse=REUSE, label=name)

## The symmetric table

Every row now chosen the same way, on data it is not scored on. The last column is what changed
against report 08, whose rows were selected two different ways.

In [ ]:
FLOOR = 0.06     # what 204 test items resolve (CLAUDE.md 6.4)

R08 = {'from-scratch ours': 0.6324, 'mmBERT': 0.5736, 'XLM-R': 0.4077,
       'untrained ours': 0.4034, 'untrained XLM-R': 0.3692}
if not SCRATCH_IS_CANONICAL:
    # A different checkpoint. Differencing against 0.632 would report a hardware difference and
    # a selection change as if they were one number.
    del R08['from-scratch ours']

print(f'{"model":<20}{"lr":>8}{"macro-F1":>10}{"sd":>7}   {"95% CI":<18}{"report 08":>10}{"move":>8}')
order = sorted(final, key=lambda k: -final[k]['mean'])
for name in order:
    r = final[name]
    ci = f'[{r["ci"][0]:.3f}, {r["ci"][1]:.3f}]'
    old = R08.get(name)
    cmp = f'{old:>10.3f}{r["mean"] - old:>+8.3f}' if old else f'{"n/a":>10}{"--":>8}'
    print(f'{name:<20}{r["lr"]:>8.0e}{r["mean"]:>10.3f}{r["sd"]:>7.3f}   {ci:<18}{cmp}')
if scratch and not SCRATCH_IS_CANONICAL:
    print('\nfrom-scratch was retrained on this runtime, so it has no report-08 comparison: '
          'that row\nmeasured a different checkpoint. The within-table gaps below are still valid.')
if not scratch:
    print('\n*** PARTIAL: the from-scratch arm is missing (no checkpoint on this runtime). ***\n'
          '    Complete here: XLM-R against its own untrained control, symmetric and dev-selected.\n'
          '    Still open:    ours against mmBERT. Do not quote a headline from this table.')

def gap(a, b):
    """Difference, with both gates the project requires: the floor and CI overlap."""
    ra, rb = final[a], final[b]
    d = ra['mean'] - rb['mean']
    overlap = ra['ci'][0] <= rb['ci'][1] and rb['ci'][0] <= ra['ci'][1]
    verdict = ('below the 0.06 floor' if abs(d) < FLOOR else
               'CIs overlap' if overlap else 'clears both gates')
    print(f'{a} - {b}: {d:+.3f}  ({verdict})')

print()
for a, b in [('from-scratch ours', 'mmBERT'),
             ('XLM-R', 'untrained XLM-R'),
             ('from-scratch ours', 'untrained ours')]:
    if a in final and b in final:
        gap(a, b)

## Reading it

Three things this can show, and they are not equally good for the study.

**If our model stays ahead of mmBERT.** The headline survives a symmetric comparison, and it is
no longer open to "you swept theirs and not yours". Note the margin was 0.059 in report 08 —
already *under* the 0.06 floor, so the honest wording is "ahead, not distinguishable", whatever
the sweep does to it.

**If XLM-R's +0.039 over its own control shrinks.** Expected, since only XLM-R was swept before.
It sharpens report 08's claim rather than damaging it: whatever XLM-R learned from 100 languages
does not reach Yoruba. If it shrinks to nothing, say that.

**If our model's win depends on the learning rate it was given.** The uncomfortable outcome, and
the reason to run this rather than assume. A default inherited from `FT_LR_SCRATCH` deciding the
study's headline would be the same failure this project has now caught ten times — a constant
chosen for one context deciding a result in another. Better found here than on the poster.

Whatever comes out, the dev-selected numbers are the ones to quote, and report 08's table needs
the learning rate and the selection rule stated next to every row.

In [ ]:
session.save_results()   # asserts Drive is really mounted before copying